# 9 · Case study — environmental impact reports (*Miljørapport*)

A real-world process modeled in Conductor: the automated *miljørapport* pipeline
that the Danish Planning and Rural Development Agency (PLST) uses to generate
environmental impact reports for energy parks. The original system is ~50 Python
modules glued together with `asyncio.create_task` + `ThreadPoolExecutor`. This
notebook captures the same shape with Conductor primitives — node registration,
edges, **shared references**, **for-each regions**, **decision nodes with edge
guards**, and a **human-in-the-loop pause**.

The pipeline does:

1. **Parse** the user's *afgrænsningsnotat* (PDF/DOCX scoping document).
2. **Extract** project context with an LLM (cached on the document hash).
3. **Fetch** external data from EA-Tools / Plandata / Miljøstyrelsen — parallel
   fan-out over `LookupSpec`s, deduplicated across chapters.
4. **Generate** 14 chapters: 3 deterministic templates + 10 LLM chapters + 1
   cumulative chapter that depends on all the others. Resume runs after.
5. **Evaluate + revise** each LLM chapter — structural checks + LLM judges
   feed a decision; failures route to a revision call.
6. **Ground** each chapter (claim → source markdown table).
7. **Assemble** the final docx with map and source appendices.

We model this with stubbed services so the notebook runs with no API keys.
The point is the *shape* — which Conductor primitive maps to which stage.


## 1 · Stub services

A handful of fakes stand in for the real LLM, EA-Tools and PDF clients so the
notebook runs offline. They're deterministic — same input, same output —
which makes the example reproducible.


In [ ]:
from __future__ import annotations

import hashlib
import textwrap
import time
from typing import Annotated, Any

# Fake LLM that returns deterministic prose seeded by the prompt hash.
def fake_llm(prompt: str, *, model: str = "gpt-stub") -> str:
    digest = hashlib.sha1(prompt.encode()).hexdigest()[:6]
    head = prompt.splitlines()[0][:60].strip() if prompt else "(empty)"
    return f"[{model}/{digest}] {head} — paragraph generated from prompt."


# Fake EA-Tools / EA ESDH lookup client.
def fake_ea_lookup(spec: "LookupSpec") -> str:
    return (
        f"=== EA report (template={spec.template_id}, "
        f"buffer={spec.buffer_m} m, label={spec.label!r}) ===\n"
        f"3 layers intersect the buffer; nearest § 3 site is 142 m N."
    )


# Fake PDF/DOCX parser.
def fake_parse_document(blob: bytes) -> str:
    return textwrap.dedent(
        '''\
        AFGRÆNSNINGSNOTAT — Energypark Demo

        The project is a 120 MW solar park near Borris, west Jutland.
        Total area: 240 ha. Distance to nearest Natura 2000 site: 6.2 km.
        Local planning: subject to municipal kommuneplantillæg.
        ...
        '''
    )


## 2 · Domain types

Mirroring the envir backend's `chapter_types.py` and `lookup_data_config.py`,
trimmed to the bits the example exercises.


In [ ]:
from pydantic import BaseModel


class LookupSpec(BaseModel):
    template_id: str
    buffer_m: int
    label: str


class ChapterDraft(BaseModel):
    chapter_id: str
    title: str
    markdown: str
    sections: dict[str, str] = {}


class EvalResult(BaseModel):
    chapter_id: str
    score: float            # 0.0 (broken) … 1.0 (clean pass)
    issues: list[str] = []


# Trimmed chapter list — full system has 14.
TEMPLATE_CHAPTERS = [
    ("ch1", "1. Indledning"),
    ("ch2", "2. Bekendtgørelsens indhold"),
    ("ch3", "3. Afgrænsning"),
]
LLM_CHAPTERS = [
    ("ch4", "4. Landskab"),
    ("ch6", "6. Jord"),
    ("ch9", "9. Vand"),
    ("ch10", "10. Biologisk mangfoldighed"),
]
CUMULATIVE_CHAPTER = ("ch14", "14. Kumulative effekter")

# Static per-chapter EA spec config (envir's CHAPTER_LOOKUPS, condensed).
LOOKUP_CONFIG: dict[str, list[LookupSpec]] = {
    "ch4":  [LookupSpec(template_id="landskab",   buffer_m=1_000,  label="Landskab og beskyttelseslinjer")],
    "ch6":  [LookupSpec(template_id="jord",       buffer_m=100,    label="Jordforhold")],
    "ch9":  [LookupSpec(template_id="vand",       buffer_m=1_000,  label="Vandløb og søer")],
    "ch10": [LookupSpec(template_id="natura2000", buffer_m=10_000, label="Arter og beskyttede naturtyper")],
}


## 3 · Register the conductor nodes

Each node wraps one stage. Function-based registration; widget annotations
double as input metadata for any frontend that reads the registry.


In [ ]:
import conductor_nodes
from conductor import GraphEdge, GraphNode, NodeRegistry, compile
from conductor.compound.for_each import FOR_EACH
from conductor.errors import FlowPausedException, HumanInputRequired
from conductor.execution.engine import collect, execute, resume
from conductor.widgets import Output, Text, Textarea

registry = NodeRegistry()

# Standard for-each markers (used in the parallel fan-out section).
conductor_nodes.loop.register(registry)


@registry.node("parse-scoping", version=1,
               name="Parse scoping note",
               description="PDF/DOCX → plain text")
def parse_scoping(
    blob_id: Annotated[str, Text(label="Document id")],
) -> Annotated[str, Output(label="Text")]:
    # Stand-in for PDF/DOCX extraction. In production the bytes come from
    # an upload; we look up by id for notebook reproducibility.
    return fake_parse_document(blob_id.encode())


@registry.node("extract-context", version=1,
               name="Extract project context",
               description="LLM extracts a one-paragraph project summary",
               idempotency_key='"ctx-" + string(size(scoping_text))')
def extract_context(
    scoping_text: Annotated[str, Textarea(label="Scoping text")],
    idempotency_key: str | None = None,
) -> Annotated[str, Output(label="Context")]:
    # idempotency_key (CEL-evaluated) survives retries — wired but unused here.
    return fake_llm(f"Summarize this project:\n{scoping_text}", model="gpt-light")


# Deduplicated EA specs as plain dicts — used as static data on
# `for-each-start.items` so the engine knows the iteration count up front.
def unique_lookup_specs() -> list[dict[str, Any]]:
    seen: set[tuple[str, int]] = set()
    out: list[dict[str, Any]] = []
    for specs in LOOKUP_CONFIG.values():
        for s in specs:
            key = (s.template_id, s.buffer_m)
            if key not in seen:
                seen.add(key)
                out.append(s.model_dump())
    return out


@registry.node("fetch-ea", version=1,
               name="Fetch EA spec",
               description="One EA-Tools call (parallel inside for-each)",
               max_retries=2, retry_delay=0.1)
def fetch_ea(
    spec: Annotated[LookupSpec, Text(label="Spec")],
) -> Annotated[dict[str, str], Output(label="Result")]:
    # Conductor stores all values as plain dicts on the wire; pydantic
    # validates on the way in but the function receives the dict form.
    # Reconstitute the model for dot access:
    spec = LookupSpec.model_validate(spec)
    # Real client would submit + poll + download. We sleep briefly
    # so parallelism is visible in node_start/node_complete timing.
    time.sleep(0.05)
    return {"key": f"{spec.template_id}@{spec.buffer_m}", "text": fake_ea_lookup(spec)}


@registry.node("index-results", version=1,
               name="Index EA results",
               description="list[ea_result] → dict[spec_key → text]")
def index_results(
    results: Annotated[list[dict[str, str]], Text(label="Results")],
) -> Annotated[dict[str, str], Output(label="Index")]:
    return {r["key"]: r["text"] for r in results}


@registry.node("template-chapter", version=1,
               name="Template chapter",
               description="Deterministic template fill (chapters 1–3)")
def template_chapter(
    chapter_id: Annotated[str, Text(label="Chapter id")],
    title: Annotated[str, Text(label="Title")],
    scoping_text: Annotated[str, Textarea(label="Scoping text")],
) -> Annotated[ChapterDraft, Output(label="Draft")]:
    md_text = f"## {title}\n\nScoped from notatet: {scoping_text.splitlines()[0]}"
    return ChapterDraft(chapter_id=chapter_id, title=title, markdown=md_text,
                        sections={"body": md_text})


@registry.node("llm-chapter", version=1,
               name="LLM chapter",
               description="Heavy LLM call (chapters 4–13)",
               max_retries=1, retry_delay=0.1)
def llm_chapter(
    chapter_id: Annotated[str, Text(label="Chapter id")],
    title: Annotated[str, Text(label="Title")],
    project_context: Annotated[str, Textarea(label="Project context")],
    ea_index: Annotated[dict[str, str], Text(label="EA index")],
) -> Annotated[ChapterDraft, Output(label="Draft")]:
    spec_keys = [f"{s.template_id}@{s.buffer_m}" for s in LOOKUP_CONFIG.get(chapter_id, [])]
    ea_text = "\n".join(ea_index.get(k, "(no data)") for k in spec_keys)
    prompt = f"Generate chapter {title}.\nContext:\n{project_context}\nData:\n{ea_text}"
    body = fake_llm(prompt, model="gpt-heavy")
    sections = {
        "legal_basis":        f"Legal basis paragraph for {title}.",
        "existing_conditions": body,
        "assessment":         f"Assessment paragraph for {title}.",
        "conclusion":         f"Conclusion: minor impact for {title}.",
    }
    md_text = f"## {title}\n\n" + "\n\n".join(sections.values())
    return ChapterDraft(chapter_id=chapter_id, title=title, markdown=md_text,
                        sections=sections)


@registry.node("evaluate", version=1,
               name="Evaluate chapter",
               description="Structural + judge LLMs → score")
def evaluate(
    draft: Annotated[ChapterDraft, Text(label="Draft")],
) -> Annotated[EvalResult, Output(label="Eval")]:
    draft = ChapterDraft.model_validate(draft)
    issues: list[str] = []
    if "legal_basis" not in draft.sections:
        issues.append("missing legal_basis")
    if len(draft.markdown) < 80:
        issues.append("too short")
    # In production this is structural + 7 parallel LLM judges.
    score = 1.0 if not issues else max(0.0, 1.0 - 0.3 * len(issues))
    return EvalResult(chapter_id=draft.chapter_id, score=score, issues=issues)


from conductor._sentinel import SKIPPED


# Router node using the SKIPPED-sentinel pattern. The decision-node /
# edge-guard feature is great for 1:1 routing (gate one outgoing edge by
# CEL), but here we need to route TWO values together — `draft` and
# `eval_result` should both reach `rev`, or neither. The SKIPPED sentinel
# composes naturally: each output handle is either a real value or
# SKIPPED, and downstream nodes whose inputs are all SKIPPED are skipped.
@registry.node("route-by-score", version=1,
               name="Route by score",
               description="Splits the chapter into revise/pass-through branches")
def route_by_score(
    draft: Annotated[ChapterDraft, Text(label="Draft")],
    eval_result: Annotated[EvalResult, Text(label="Eval")],
) -> tuple[
    Annotated[Any, Output(label="Draft → revise")],
    Annotated[Any, Output(label="Eval → revise")],
    Annotated[Any, Output(label="Draft → pass-through")],
]:
    eval_result = EvalResult.model_validate(eval_result)
    if eval_result.score < 0.7:
        return draft, eval_result.model_dump(), SKIPPED
    return SKIPPED, SKIPPED, draft


@registry.node("revise", version=1,
               name="Revise chapter",
               description="LLM revision call",
               max_retries=2, retry_delay=0.1)
def revise(
    draft: Annotated[ChapterDraft, Text(label="Draft")],
    eval_result: Annotated[EvalResult, Text(label="Eval")],
) -> Annotated[ChapterDraft, Output(label="Revised")]:
    draft = ChapterDraft.model_validate(draft)
    eval_result = EvalResult.model_validate(eval_result)
    feedback = "; ".join(eval_result.issues) or "(none)"
    prompt = f"Fix these issues in {draft.title}: {feedback}\n\n{draft.markdown}"
    revised_body = fake_llm(prompt, model="gpt-heavy")
    sections = {**draft.sections, "existing_conditions": revised_body}
    md_text = f"## {draft.title}\n\n" + "\n\n".join(sections.values())
    return ChapterDraft(chapter_id=draft.chapter_id, title=draft.title,
                        markdown=md_text, sections=sections)


@registry.node("ground", version=1,
               name="Ground chapter",
               description="Claim → source table")
def ground(
    draft: Annotated[ChapterDraft, Text(label="Draft")],
) -> Annotated[ChapterDraft, Output(label="Grounded")]:
    draft = ChapterDraft.model_validate(draft)
    table = (
        "\n\n### Sources\n"
        "| Claim | Source |\n"
        "|---|---|\n"
        f"| (extracted by judge) | {draft.title} dataset |\n"
    )
    return ChapterDraft(
        chapter_id=draft.chapter_id, title=draft.title,
        markdown=draft.markdown + table,
        sections={**draft.sections, "sources": table},
    )


@registry.node("cumulative", version=1,
               name="Cumulative chapter",
               description="Chapter 14 — depends on all earlier chapters")
def cumulative(
    title: Annotated[str, Text(label="Title")],
    earlier_chapters: Annotated[list[ChapterDraft], Text(label="Earlier chapters")],
) -> Annotated[ChapterDraft, Output(label="Draft")]:
    earlier_chapters = [ChapterDraft.model_validate(c) for c in earlier_chapters]
    bullets = "\n".join(f"- {c.title}: minor cumulative contribution" for c in earlier_chapters)
    md_text = f"## {title}\n\n{bullets}"
    return ChapterDraft(chapter_id="ch14", title=title, markdown=md_text,
                        sections={"body": md_text})


@registry.node("resume", version=1,
               name="Resume",
               description="Citizen-friendly summary across all chapters")
def resume_node(
    chapters: Annotated[list[ChapterDraft], Text(label="All chapters")],
) -> Annotated[str, Output(label="Resume markdown")]:
    chapters = [ChapterDraft.model_validate(c) for c in chapters]
    titles = ", ".join(c.title for c in chapters)
    return fake_llm(f"Plain-language summary of: {titles}", model="gpt-light")


@registry.node("assemble-docx", version=1,
               name="Assemble docx",
               description="Stitches chapters + appendices into final artifact")
def assemble_docx(
    chapters: Annotated[list[ChapterDraft], Text(label="Chapters")],
    resume_text: Annotated[str, Textarea(label="Resume")],
) -> Annotated[dict[str, Any], Output(label="Artifact")]:
    chapters = [ChapterDraft.model_validate(c) for c in chapters]
    body = "\n\n".join(c.markdown for c in chapters) + "\n\n## Resume\n\n" + resume_text
    return {"filename": "miljorapport.docx", "size_bytes": len(body), "preview": body[:200]}


print(f"Registered {len(registry.all())} nodes.")


## 4 · Layer 1 — parallel external fetches via `for-each`

The original `external_data_orchestrator.py` runs N specs through a
`ThreadPoolExecutor`. In Conductor it's a for-each region: pre-build the
deduplicated spec list, hand it to `for-each-start.items` as **static
data** on the node (the canonical way to seed a for-each — `items` uses
the `ConnectionList` widget, which would otherwise wrap a single edge
source into a one-key dict instead of unpacking it). The body fetches
one spec; `for-each-end` collects.

Independent fetches overlap automatically — Conductor schedules each
iteration as soon as it's ready, with sync calls offloaded to
`asyncio.to_thread`. Setting `execution_mode="Parallel"` on the start
node enables thread-pool parallelism inside the region.


In [ ]:
fetch_nodes = [
    GraphNode("start", "for-each-start@1",
                  {"items": unique_lookup_specs(), "execution_mode": "Parallel"}),
    GraphNode("body",  "fetch-ea@1", None),
    GraphNode("end",   "for-each-end@1", None),
    GraphNode("index", "index-results@1", None),
]
fetch_edges = [
    GraphEdge("e2", "start", "body",  "output_1", "spec"),
    GraphEdge("e3", "body",  "end",   "result", "item"),
    GraphEdge("e4", "end",   "index", "result", "results"),
]

t0 = time.perf_counter()
fetch_compiled = compile(
    nodes=fetch_nodes, edges=fetch_edges, registry=registry,
    compound_types=[FOR_EACH],
)
fetch_results = await collect(execute(fetch_compiled))
print(f"Fetched {len(fetch_results['index']['result'])} unique specs "
      f"in {time.perf_counter() - t0:.2f}s")
for k, v in fetch_results["index"]["result"].items():
    print(f"  {k:24s} -> {v.splitlines()[0]}")


## 5 · Layer 2 — per-chapter pipeline with SKIPPED-sentinel routing

Each LLM chapter goes through `generate → evaluate → route → revise/skip
→ ground`. The router emits two parallel branches (revise / pass-through)
using the **SKIPPED sentinel** — the canonical Conductor pattern for
conditional flow when downstream nodes need multiple inputs from the
same branch.

```
  llm-chapter ──┬─> evaluate ──┐
                │              ▼
                └──────────> route-by-score
                              ├─(score < 0.7)──> revise ──┐
                              └─(else)──────────────────────> ground
                                                              ▲
                                              (rev or route.output_3 — only one is live)
```

`route-by-score` returns a 3-tuple: `(draft_for_revise, eval_for_revise,
draft_for_passthrough)`. Two of the three are always `SKIPPED`; the
resolver picks the live one when an input has multiple incoming edges
where all but one are SKIPPED. `revise` is auto-skipped when both its
inputs are SKIPPED.

> Decision nodes + edge guards (`is_decision=True`, `when=` on edges)
> are the right tool when one routed value goes to one downstream node.
> When two routed values must travel together, SKIPPED-sentinel routing
> composes cleanly — that's what we use here.


In [ ]:
chapter_pipeline_nodes = [
    GraphNode("ctx",   "extract-context@1", {"scoping_text": fake_parse_document(b"x")}),
    GraphNode("start", "for-each-start@1",
                  {"items": unique_lookup_specs(), "execution_mode": "Parallel"}),
    GraphNode("body",  "fetch-ea@1", None),
    GraphNode("end",   "for-each-end@1", None),
    GraphNode("index", "index-results@1", None,
              produces={"result": "EA index"}),
    GraphNode("gen",   "llm-chapter@1", {"chapter_id": "ch4", "title": "4. Landskab"},
              consumes={"ea_index": ("index", "result")}),
    GraphNode("eval",  "evaluate@1", None),
    GraphNode("route", "route-by-score@1", None),
    GraphNode("rev",   "revise@1", None),
    GraphNode("grnd",  "ground@1", None),
]
chapter_pipeline_edges = [
    # external fetch fan-out
    GraphEdge("e2", "start", "body",  "output_1", "spec"),
    GraphEdge("e3", "body",  "end",   "result", "item"),
    GraphEdge("e4", "end",   "index", "result", "results"),
    # context → generation → eval → route
    GraphEdge("e5", "ctx",   "gen",   "result",   "project_context"),
    GraphEdge("e6", "gen",   "eval",  "result",   "draft"),
    GraphEdge("e7", "gen",   "route", "result",   "draft"),
    GraphEdge("e8", "eval",  "route", "result",   "eval_result"),
    # SKIPPED-sentinel branches.
    GraphEdge("e9",  "route", "rev",   "output_1", "draft"),
    GraphEdge("e10", "route", "rev",   "output_2", "eval_result"),
    GraphEdge("e11", "route", "grnd",  "output_3", "draft"),  # pass-through
    GraphEdge("e12", "rev",   "grnd",  "result",   "draft"),  # revise path
]

cp_compiled = compile(
    nodes=chapter_pipeline_nodes,
    edges=chapter_pipeline_edges,
    registry=registry,
    compound_types=[FOR_EACH],
)
cp_results = await collect(execute(cp_compiled))

final_draft = ChapterDraft.model_validate(cp_results["grnd"]["result"])
print(f"Final draft: {final_draft.title}  ({len(final_draft.markdown)} chars)")
print(f"Sections: {list(final_draft.sections)}")
print(f"Was revised? {'rev' in cp_results}")


## 6 · Layer 3 — the full report

Bigger flow: 3 template chapters + 4 LLM chapters + 1 cumulative + resume +
docx assembly. Two Conductor primitives keep it tidy:

- **Shared references** broadcast `project_context` and the `EA index` to
  every chapter — no fan-out edge spaghetti.
- The **cumulative chapter** consumes all earlier chapter drafts via
  ConnectionList (one edge per chapter, aggregated into a list).

For brevity we skip the eval / decide / revise / ground sub-pipeline here
— substitute the 5 nodes from §5 inline if you want it on every LLM chapter.


In [ ]:
def build_full_report() -> tuple[list[GraphNode], list[GraphEdge]]:
    nodes: list[GraphNode] = [
        # `parse` produces its scoping text as a shared reference so
        # template chapters can consume it without an explicit edge each.
        GraphNode("parse", "parse-scoping@1", {"blob_id": "demo"},
                  produces={"result": "scoping text"}),
        GraphNode("ctx",   "extract-context@1", None,
                  produces={"result": "project context"}),
        # External data fan-out — items pre-computed from LOOKUP_CONFIG.
        GraphNode("start", "for-each-start@1",
                  {"items": unique_lookup_specs(), "execution_mode": "Parallel"}),
        GraphNode("body",  "fetch-ea@1", None),
        GraphNode("end",   "for-each-end@1", None),
        GraphNode("index", "index-results@1", None,
                  produces={"result": "EA index"}),
    ]
    edges: list[GraphEdge] = [
        GraphEdge("e0", "parse", "ctx",   "result", "scoping_text"),
        GraphEdge("e2", "start", "body",  "output_1", "spec"),
        GraphEdge("e3", "body",  "end",   "result", "item"),
        GraphEdge("e4", "end",   "index", "result", "results"),
    ]

    chapter_node_ids: list[str] = []
    for cid, title in TEMPLATE_CHAPTERS:
        nodes.append(
            GraphNode(cid, "template-chapter@1",
                      {"chapter_id": cid, "title": title},
                      consumes={"scoping_text": ("parse", "result")})
        )
        chapter_node_ids.append(cid)

    # LLM chapters (4–13, condensed to 4). Both project context and EA
    # index are consumed via shared reference — broadcast, no edges.
    for cid, title in LLM_CHAPTERS:
        nodes.append(
            GraphNode(cid, "llm-chapter@1",
                      {"chapter_id": cid, "title": title},
                      consumes={
                          "project_context": ("ctx",   "result"),
                          "ea_index":        ("index", "result"),
                      })
        )
        chapter_node_ids.append(cid)

    # Chapter 14 — cumulative — depends on all earlier chapter drafts.
    cum_id, cum_title = CUMULATIVE_CHAPTER
    nodes.append(GraphNode(cum_id, "cumulative@1", {"title": cum_title}))
    for i, src in enumerate(chapter_node_ids):
        edges.append(GraphEdge(f"cum_{i}", src, cum_id, "result", "earlier_chapters"))
    chapter_node_ids.append(cum_id)

    # Resume — depends on all chapters including ch14.
    nodes.append(GraphNode("resume", "resume@1", None))
    for i, src in enumerate(chapter_node_ids):
        edges.append(GraphEdge(f"res_{i}", src, "resume", "result", "chapters"))

    # Final docx assembly.
    nodes.append(GraphNode("docx", "assemble-docx@1", None))
    for i, src in enumerate(chapter_node_ids):
        edges.append(GraphEdge(f"asm_{i}", src, "docx", "result", "chapters"))
    edges.append(GraphEdge("asm_resume", "resume", "docx", "result", "resume_text"))

    return nodes, edges


fr_nodes, fr_edges = build_full_report()
fr_compiled = compile(
    nodes=fr_nodes, edges=fr_edges, registry=registry, compound_types=[FOR_EACH],
)
print(f"Full report flow: {len(fr_nodes)} nodes, {len(fr_edges)} edges, "
      f"compiled in {len(fr_compiled.execution_order)} steps")
print(f"Type warnings: {len(fr_compiled.type_warnings)}")

t0 = time.perf_counter()
fr_results = await collect(execute(fr_compiled))
print(f"Full report executed in {time.perf_counter() - t0:.2f}s")

artifact = fr_results["docx"]["result"]
print(f"\nArtifact: {artifact['filename']} ({artifact['size_bytes']} bytes)")
print("Preview:\n" + artifact["preview"])


## 7 · Visualizing the graph

`conductor.viz.to_mermaid()` renders any `CompiledGraph` as a Mermaid
flowchart. Wrapped in a fenced code block, GitHub renders it natively;
in Jupyter we feed it through `IPython.display.Markdown`. Visual
conventions: decision diamonds, compound subroutines, managed
parallelograms, dashed arrows for shared references.


In [ ]:
from conductor.viz import to_mermaid
from IPython.display import Markdown

mermaid_src = to_mermaid(fr_compiled, direction="LR")
Markdown(f"```mermaid\n{mermaid_src}\n```")


## 8 · Variation — human-in-the-loop dataset-mapping review

In the real system, between the overlap analysis and the full-report run a
human edits the dataset → chapter assignments in the DB. We model that as a
`HumanInputRequired` pause: the flow stops, hands the proposed mapping out
as a checkpoint, and resumes once a reviewer responds.

Same pattern as `06_human_in_the_loop.ipynb`, applied to the envir process.


In [ ]:
@registry.node("propose-mapping", version=1,
               name="Propose mapping",
               description="Suggested dataset → chapter assignments")
def propose_mapping(
    scoping_text: Annotated[str, Textarea(label="Scoping text")],
) -> Annotated[dict[str, list[str]], Output(label="Mapping")]:
    return {
        "ch4":  ["landskab", "beskyttelseslinjer"],
        "ch6":  ["jordforurening"],
        "ch9":  ["vandloeb", "soeer"],
        "ch10": ["natura2000", "habitatkort"],
    }


@registry.node("review-mapping", version=1,
               name="Review mapping",
               description="Pause for human review/edit",
               actor={"kind": "human", "role": "case_officer"})
def review_mapping(
    mapping: Annotated[dict[str, list[str]], Text(label="Proposed mapping")],
) -> Annotated[dict[str, list[str]], Output(label="Approved mapping")]:
    raise HumanInputRequired(
        prompt="Approve or edit the dataset → chapter mapping below.",
        schema={"mapping": "dict[str, list[str]]"},
    )


@registry.node("apply-mapping", version=1,
               name="Apply mapping",
               description="Persists the approved mapping")
def apply_mapping(
    mapping: Annotated[dict[str, list[str]], Text(label="Approved mapping")],
) -> Annotated[str, Output(label="Status")]:
    return f"applied {len(mapping)} chapter assignments"


hitl_nodes = [
    GraphNode("parse", "parse-scoping@1", {"blob_id": "demo"}),
    GraphNode("prop",  "propose-mapping@1", None),
    GraphNode("rev",   "review-mapping@1", None),
    GraphNode("apply", "apply-mapping@1", None),
]
hitl_edges = [
    GraphEdge("e1", "parse", "prop",  "result", "scoping_text"),
    GraphEdge("e2", "prop",  "rev",   "result", "mapping"),
    GraphEdge("e3", "rev",   "apply", "result", "mapping"),
]
hitl_compiled = compile(nodes=hitl_nodes, edges=hitl_edges, registry=registry)

# Phase 1 — run until pause.
import json
try:
    await collect(execute(hitl_compiled))
    print("ERROR: should have paused")
except FlowPausedException as e:
    checkpoint = e.checkpoint
    print(f"Paused at: {checkpoint['waiting_node_id']}")
    print(f"Prompt: {checkpoint['prompt']}")

# Phase 2 — resume with the human's edited mapping.
edited_mapping = {
    "ch4":  ["landskab"],                   # reviewer dropped one
    "ch6":  ["jordforurening", "raastof"],  # reviewer added one
    "ch9":  ["vandloeb", "soeer"],
    "ch10": ["natura2000", "habitatkort"],
}
resumed = await collect(resume(hitl_compiled, json.loads(json.dumps(checkpoint)),
                               response=edited_mapping))
print(f"Resumed: {resumed['apply']['result']}")


## 9 · What we modeled

| Envir module / function | Conductor primitive |
|---|---|
| `pdf_extraction_service.py` | `parse-scoping` node |
| `chapter_generation_service.py:_extract_project_context` | `extract-context` node + `idempotency_key` (matches the envir `HashCache`) |
| `external_data_orchestrator.run_lookups` | `for-each` region: `unique_lookup_specs()` → `fetch-ea` body → `index-results` |
| `lookup_data_config.py:CHAPTER_LOOKUPS` | `LOOKUP_CONFIG` dict consulted by `llm-chapter` |
| `_generate_template_chapter` | `template-chapter` node |
| `_generate_llm_chapter` | `llm-chapter` node, with `project_context` and `ea_index` pulled in via **shared references** (no edges crossing the parallel chapter wave) |
| `revision_service.evaluate_chapter` | `evaluate` node |
| `revision_service.revise_chapter` | `route-by-score` SKIPPED-sentinel router + `revise` node (revise path runs only when `eval.score < 0.7`) |
| `revision_service.ground_chapter` | `ground` node |
| `_generate_cumulative_chapter` | `cumulative` node, depends on all chapters 1–13 via `ConnectionList`-style edge fan-in |
| `_generate_resume` | `resume` node |
| `docx_export_service.markdown_to_docx` | `assemble-docx` node |
| Dataset → chapter mapping review (DB step) | `review-mapping` node raising `HumanInputRequired` |
| `agent_service` LLM split-completion fallback | per-node `max_retries=1` + a manual draft-fallback node would model it; not exercised here |

Things deliberately out of scope of the example so it stays readable:

- The 7 parallel LLM judges inside `evaluate` (would be a sub-flow of its own).
- Map embedding (`map_embedding.replace_map_markers`) — pure transform; one
  more node, nothing structurally new.
- The `revise-stream` POST endpoint — same shape as the HITL section above.
